## Part 1: Dataset Preparation and Fine-Tuning (7 points)
### Step 1: Download the IMDB Dataset (1 point)

*   Use the IMDB dataset from Kaggle: /kaggle/input/imdb-dataset/IMDB
Dataset.csv.
*    Load the dataset using Pandas and verify it in your notebook.





In [2]:
# Install Kaggle API
!pip install kaggle

In [3]:
# Copy kaggle.json from Google Drive to Colab's working directory
import os
# Other needed imports
import pandas as pd
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Define the path where your kaggle.json is stored in Google Drive
kaggle_json_path = "/content/drive/My Drive/Colab Notebooks/kaggle.json"

# Create a .kaggle folder and move kaggle.json there
!mkdir -p ~/.kaggle
!cp "{kaggle_json_path}" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json  # Set permissions

# Verify Kaggle installation. This show only some of the datasets. There is more
# at https://www.kaggle.com/datasets
!kaggle datasets list

ref                                                           title                                              size  lastUpdated          downloadCount  voteCount  usabilityRating  
------------------------------------------------------------  ------------------------------------------------  -----  -------------------  -------------  ---------  ---------------  
anandshaw2001/netflix-movies-and-tv-shows                     Netflix Movies and TV Shows                         1MB  2025-01-03 10:33:01          14194        368  1.0              
asinow/car-price-dataset                                      Car Price Dataset                                 135KB  2025-01-26 19:53:28           3672         51  1.0              
ankushpanday1/alzheimers-prediction-dataset-global            Alzheimer’s Prediction Dataset (Global)             1MB  2025-01-30 14:38:39           1536         33  1.0              
asinow/airplane-price-dataset                                 Airplane Price Dat

In [5]:
# Download the IMDB dataset
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

# Unzip the dataset
!unzip imdb-dataset-of-50k-movie-reviews.zip

# Load the dataset using Pandas
df = pd.read_csv("IMDB Dataset.csv")

# Display the dataset
print("IMDB Dataset Loaded Successfully!")
print(df)


Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
 89% 23.0M/25.7M [00:00<00:00, 119MB/s]
100% 25.7M/25.7M [00:00<00:00, 115MB/s]
Archive:  imdb-dataset-of-50k-movie-reviews.zip
  inflating: IMDB Dataset.csv        
IMDB Dataset Loaded Successfully!
                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going 

### Step 2:  Data Preprocessing (1 point)

1. Clean and preprocess the dataset:
  *   Encode the sentiment column (positive -> 1, negative -> 0).
  *   Retain only the review and label columns.
2. Split the data into training and validation, testing

In [6]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 15.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_syste

In [7]:
from datasets import Dataset

In [8]:
# Ensure sentiment column has no spaces and is in lowercase
df['sentiment'] = df['sentiment'].str.strip().str.lower()

# Encode the sentiment column (positive -> 1, negative -> 0)
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Check if any NaN values remain
print(df['sentiment'].isna().sum())  # Should print 0 if everything is mapped correctly

0


In [9]:
# Convert Pandas DataFrame to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df)

# Split dataset using Hugging Face's built-in method
train_test_split = hf_dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)
train_dataset = train_test_split["train"]
temp_dataset = train_test_split["test"]

# Further split into validation (10%) and test (10%)
val_test_split = temp_dataset.train_test_split(test_size=0.5, shuffle=True, seed=42)
val_dataset = val_test_split["train"]
test_dataset = val_test_split["test"]

# Print dataset sizes
print(f"Training set: {len(train_dataset)} samples")
print(f"Validation set: {len(val_dataset)} samples")
print(f"Testing set: {len(test_dataset)} samples")

# Display a few samples
train_dataset.to_pandas().head()

Training set: 40000 samples
Validation set: 5000 samples
Testing set: 5000 samples


,review,sentiment
0,The film disappointed me for many reasons: fir...,0
1,"Can this ""film"" be considered as a film? Imagi...",0
2,not really sure what to make of this movie. ve...,1
3,"Wow, I forgot how great this movie was until I...",1
4,"Produced by International Playhouse Pictures, ...",0


### Step 3: Model Selection and Tokenization (1 point)

1. Select a pre-trained Hugging Face transformer model for fine-tuning (e.g.,
distilbert-base-uncased).
2. Tokenize the dataset with (see if required )
  *   Truncation.
  *   Padding
  *   Maximum sequence length of 256.


In [10]:
from transformers import AutoTokenizer, DataCollatorWithPadding

# Load the DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization function with dynamic padding
def tokenize_data(example):
    encoding = tokenizer(
        example["review"],  # Text column
        padding=False,  # Dynamic padding
        truncation=True,
        max_length=256
    )
    encoding["labels"] = example["sentiment"]  # Add labels
    return encoding

# Apply tokenization to each dataset
train_dataset = train_dataset.map(tokenize_data, batched=True)
val_dataset = val_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

# Use DataCollatorWithPadding for dynamic batching
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Verify tokenized output
print(f"Sample Tokenized Input IDs: {train_dataset[0]['input_ids']}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Sample Tokenized Input IDs: [101, 1996, 2143, 9364, 2033, 2005, 2116, 4436, 1024, 2034, 1997, 2035, 1996, 15921, 1997, 1037, 2925, 2029, 2790, 2012, 2034, 12689, 2000, 2033, 2001, 2092, 1011, 2328, 2021, 2106, 2069, 3444, 1037, 14785, 2535, 1012, 2059, 1010, 1996, 2466, 2993, 2001, 1037, 5410, 6100, 1997, 2439, 1999, 5449, 1012, 1996, 2690, 1011, 2789, 4292, 1010, 2158, 2007, 2155, 6010, 2047, 2611, 6931, 1010, 13173, 11045, 3347, 1010, 1996, 4950, 5750, 1998, 1996, 13425, 1011, 2035, 2008, 2001, 1037, 2200, 2919, 20017, 1997, 1996, 6581, 2439, 1999, 5449, 2029, 2018, 2036, 21553, 1012, 2023, 3185, 5363, 2000, 2022, 2242, 8235, 1998, 3451, 1024, 2009, 2003, 2025, 1012, 1045, 4687, 2339, 5199, 18091, 2130, 2641, 2725, 2023, 2143, 999, 1029, 1996, 2931, 2364, 3883, 2003, 9643, 1011, 2106, 2016, 2377, 1996, 3653, 3597, 2290, 1999, 7162, 3189, 1029, 1998, 2339, 2079, 2017, 2031, 2000, 2265, 1996, 12436, 20876, 1999, 1037, 3185, 2066, 2023, 1029, 2439, 1999, 5449, 2134, 1005, 1056, 2031, 20

In [11]:
# Print a sample review and its tokenized version
sample_index = 0  # Select the first sample for demonstration
sample_text = train_dataset[sample_index]["review"]
sample_tokens = train_dataset[sample_index]["input_ids"]
sample_attention_mask = train_dataset[sample_index]["attention_mask"]

print("Original Review:")
print(sample_text)
print("\nTokenized Input IDs:")
print(sample_tokens)
print("\nAttention Mask:")
print(sample_attention_mask)

Original Review:
The film disappointed me for many reasons: first of all the depiction of a future which seemed at first realistic to me was well-built but did only feature a marginal role. Then, the story itself was a weak copy of Lost in Translation. The Middle-Eastern setting, man with family meets new girl overseas, karaoke bar, the camera movements and the imagery - all that was a very bad imitation of the excellent Lost in Translation which had also credibility. This movie tries to be something brilliant and cultural: it is not. I wonder why Tim Robbins even considered doing this film!? The female main actress is awful - did she play the precog in Minority Report? And why do you have to show the vagina in a movie like this? Lost in Translation didn't have to show excessive love scenes. R-Rated just for this? This movie isn't even worth watching it from a videostore!

Tokenized Input IDs:
[101, 1996, 2143, 9364, 2033, 2005, 2116, 4436, 1024, 2034, 1997, 2035, 1996, 15921, 1997, 10

### Step 4: Fine-Tune the Model (2 points)

1.  Fine-tune the model on the IMDB dataset for 2 epochs using the Hugging Face
Trainer.
2. Set training parameters:
  *   Learning rate: 5e-5 or your own
  *   Batch size: 16 or 32
  *   Evaluation at the end of each epoch.
3.  Ensure that metrics like accuracy, precision, recall, and F1-score are logged during training

In [12]:
import torch
import numpy as np
from transformers import (AutoModelForSequenceClassification, Trainer, TrainingArguments)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
os.environ["WANDB_DISABLED"] = "true"  # Disables Weights & Biases


In [13]:
# Load the DistilBERT model for binary classification (2 classes: positive/negative)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)  # Get predicted class
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# Set up training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",  # Evaluate at the end of each epoch
    save_strategy="epoch",  # Save model at each epoch
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,  # Fine-tune for 2 epochs
    weight_decay=0.01,
    logging_dir="./logs",
    logging_strategy="epoch",
    load_best_model_at_end=True
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,  # Handles dynamic padding
    compute_metrics=compute_metrics
)

# Fine-tune the model
trainer.train()


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
<ipython-input-13-67a67407ce3a>:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.287700,0.210833,0.915800,0.930193,0.896636,0.913106
2,0.142800,0.245992,0.924400,0.923732,0.922983,0.923358


TrainOutput(global_step=5000, training_loss=0.21524685668945312, metrics={'train_runtime': 1800.3464, 'train_samples_per_second': 44.436, 'train_steps_per_second': 2.777, 'total_flos': 5298695946240000.0, 'train_loss': 0.21524685668945312, 'epoch': 2.0})

In [23]:
test_results = trainer.evaluate(test_dataset)
print(test_results)


{'eval_loss': 0.2287018746137619, 'eval_accuracy': 0.9042, 'eval_precision': 0.920507399577167, 'eval_recall': 0.8820907617504052, 'eval_f1': 0.9008897165321746, 'eval_runtime': 34.0396, 'eval_samples_per_second': 146.888, 'eval_steps_per_second': 9.195, 'epoch': 2.0}


### Step 5: Save and Upload the Model to Hugging Face (2 points)

1. Save the fine-tuned model and tokenizer locally using save_pretrained().
2. Log in to Hugging Face using notebook_login.
3. Upload the model to Hugging Face using push_to_hub.
4. Verify the model on Hugging Face Hub and include the link in your notebook.

In [19]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Save the fine-tuned model correctly
model.save_pretrained("/content/drive/My Drive/fine_tuned_distilbert")
tokenizer.save_pretrained("/content/drive/My Drive/fine_tuned_distilbert")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


('/content/drive/My Drive/fine_tuned_distilbert/tokenizer_config.json',
 '/content/drive/My Drive/fine_tuned_distilbert/special_tokens_map.json',
 '/content/drive/My Drive/fine_tuned_distilbert/vocab.txt',
 '/content/drive/My Drive/fine_tuned_distilbert/added_tokens.json',
 '/content/drive/My Drive/fine_tuned_distilbert/tokenizer.json')

In [20]:
from huggingface_hub import notebook_login

# Login to Hugging Face Hub
notebook_login()


In [22]:
# Load the saved model and tokenizer from Google Drive
model_path = "/content/drive/My Drive/fine_tuned_distilbert"

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Push the model to Hugging Face Hub
model.push_to_hub("NikkeS/imdb-distilbert")
tokenizer.push_to_hub("NikkeS/imdb-distilbert")


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/NikkeS/imdb-distilbert/commit/0832db838b97bfd2a5ff374df577eedaf193b1ce', commit_message='Upload tokenizer', commit_description='', oid='0832db838b97bfd2a5ff374df577eedaf193b1ce', pr_url=None, repo_url=RepoUrl('https://huggingface.co/NikkeS/imdb-distilbert', endpoint='https://huggingface.co', repo_type='model', repo_id='NikkeS/imdb-distilbert'), pr_revision=None, pr_num=None)

Link to model: https://huggingface.co/NikkeS/imdb-distilbert

# Part 2: API Development and Testing (5 points)  
## Step 6: Set Up the Backend API (1 point)

1.    Use FastAPI or Flask, Express, Nest Nodejs to create an API.
2.    Define a POST endpoint (/analyze/) that:
      * Accepts
          * text: The input text for sentiment analysis.
          * model: A parameter specifying the model to use (custom or llama).
      * Returns:
          *  Sentiment (positive or negative).
          *  Confidence score.




## Step 7: Load Models (1 point)
    1. Load the fine-tuned model from Hugging Face.
    2. Access the Llama 3 model using the Groq Cloud API.



Create main.py with Croq api key.   
After that run in terminal: uvicorn main:app --reload

This starts FastAPI

## Step 8: Test the API Locally (1 point)
1.    Test the /analyze/ endpoint with both models (custom and llama) using:
      * Postman.
          * Worked!
      * curl.
          *  Worked!
      * Python requests


In [8]:
import requests

url = "http://127.0.0.1:8000/analyze/"
data = {
    "text": "This movie was amazing!",
    "model": "custom"
}
response = requests.post(url, json=data)

print("Response:", response.json())


Response: {'sentiment': 'positive', 'confidence': 0.99}


In [54]:
import requests

url = "http://127.0.0.1:8000/analyze/"
data = {
    "text": "This movie was amazing!",
    "model": "llama"
}
response = requests.post(url, json=data)

print("Response:", response.json())

Response: {'sentiment': 'positive', 'confidence': 0.95}


## Step 9: Define the Llama 3 Prompt (1 point)
1. Write a clear and reusable prompt for the Llama 3 model in Groq Cloud.  
       Example: can be improved more  

        "Classify the sentiment of this text as positive or negative:          'This movie was fantastic"


**Prompt that was used**

Analyze the sentiment of this text: `{request.text}`.  
Respond in JSON format like this: `{"sentiment": "positive", "confidence": 0.92}`.  
Sentiment can only be positive/negative, nothing else.  
Analyze sentiment and add confidence score according to that.  
Give confidence score with two digits.  

## Step 10: Test with Both Models (1 point)
1. Verify that the API works for both the fine-tuned model and the Llama 3 model.
2. Ensure the results return the sentiment score too.
3. For Groq you can add into prompt,

All working!

## Step 11: React UI Design (1 point)
* A text input field for user input.
* A dropdown menu for model selection:
  * Custom Model.
  * Llama 3.
* A button labeled "Analyze Sentiment" to send input and selected model to the backend API.
* A result display section showing:
  * Sentiment (positive or negative).
  * Confidence score(optional)
 
Done!


In [ ]:
## Step 12: Submit GitHub Repository (1 point)
1.    Upload all code (notebook, backend, and UI explanation) to a public GitHub repository.
2.    Include a README.md file that explains how to:
      *  Install dependencies.
      *  Run the notebook and API locally.
      *  Use the endpoints.

Done!

## Step 13: Record a YouTube Demo Video (1 point)
1.    Record a demo video (2-3 minutes) showing:
      * Testing the system with both models (custom and llama).
      * One question with custom fine and one with llam3 any llama 3 will be fine.
2.    Upload the video to YouTube and include the link in your notebook.

Link to video: https://youtu.be/yLKP4kVK_U4